# Volatility Dynamics 10 — physical jump model vs 0DTE option wings

**VD 019. Cross-branch reuse approved by Ivan (2026-10-10):** VRP branch Black-76 conventions (`option_pricing`) and its pricing-audit findings (VRP 006). Deep-OTM 0DTE quotes are tick-priced, so the comparison is in **price space against bid/ask**, never exchange IV. No session after the VRP confirmation boundary (2026-11-06) is used.

Each two-sided OTM 0DTE quote is priced under a compound-Poisson + diffusion model:

$$P^{jump}=\sum_{n=0}^{3}\text{Pois}(n;\Lambda)\,\mathbb E\big[\text{B76}(F e^{S_n-\Lambda(\bar m-1)},K,D)\big]$$

- $D$ is the **ex-ante** diffusive variance to expiry: backward 4h bipower × Module 08 clock, with settlement-TWAP weights.
- $\Lambda$ is the expected number of jumps.
- The jumps are measured signed jump sizes (local-σ units) × current σ.
- Jump rate and sizes are **leave-one-session-out**.

**Pre-registered (VD 019 §4):**
- **Sample** — non-floor two-sided OTM quotes, TTE 60–1200 min, 15-min grid, assets not flagged by Module 09 P2.
- **Primary** — moneyness $z=|\ln K/F|/\sqrt D\in[2,5)$; $M$ = median over sessions of the session-median $\ln(\text{mid}/P^{jump})$.
- **Verdict** — **RICH** if the session-bootstrap 95% CI > 0, **CHEAP** if < 0, else **CONSISTENT**.
- **Also reported** — all buckets, per asset, no-jump model, sensitivities (diffusion ×0.85, λ ×0.5/×2), and hold-to-settlement P&L. Outputs: `10/`.

In [ ]:
from pathlib import Path
import json, os, sys, time, warnings
import numpy as np
import pandas as pd
import duckdb
import matplotlib.pyplot as plt
from dotenv import load_dotenv

warnings.filterwarnings('ignore', category=RuntimeWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 300)
pd.set_option('display.max_rows', 300)

load_dotenv()
DATA_ROOT = Path(os.environ['DERIBIT_DATA_ROOT']).expanduser().resolve()
DB_PATH = DATA_ROOT / 'database' / 'research.duckdb'
PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / 'src' / 'data_engineering').exists():
    raise RuntimeError('Repository root not found.')
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from data_engineering.modeling import volatility_vd_b_core as vb
from data_engineering.modeling import volatility_jump_hawkes as jh
from data_engineering.modeling import volatility_jump_compound as jc
from data_engineering.modeling import volatility_jump_option_wings as ow
START = os.getenv('VD10_START', '').strip() or None
END = os.getenv('VD10_END', '').strip() or None
ASSETS = [x.strip().upper() for x in os.getenv('VD10_ASSETS', '').split(',') if x.strip()]
N_BOOT = int(os.getenv('VD10_N_BOOT', str(ow.N_BOOT)))
OUT = PROJECT_ROOT / 'notebooks' / 'Volatility_Dynamics' / '10'
OUT.mkdir(parents=True, exist_ok=True)
print('Module 10', ow.VERSION, ow.selftest())


In [ ]:
def qident(name):
    return '"' + str(name).replace('"', '""') + '"'

def load_frame(columns, start=None, end=None, assets=None):
    cols = list(dict.fromkeys(['state_time', 'underlying', 'return_5m'] + list(columns)))
    where = []
    if start:
        ts = pd.Timestamp(start)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time >= TIMESTAMPTZ '{ts.isoformat()}'")
    if end:
        ts = pd.Timestamp(end)
        ts = ts.tz_localize('UTC') if ts.tzinfo is None else ts.tz_convert('UTC')
        where.append(f"state_time < TIMESTAMPTZ '{(ts + pd.Timedelta(days=1)).isoformat()}'")
    if assets:
        where.append('underlying IN (' + ','.join(repr(x) for x in assets) + ')')
    sql = ('SELECT ' + ', '.join(qident(c) for c in cols) + ' FROM time_series_feature_state'
           + ('' if not where else ' WHERE ' + ' AND '.join(where)) + ' ORDER BY underlying, state_time')
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        df = con.execute(sql).df()
    finally:
        con.close()
    if df.empty:
        raise RuntimeError('No rows found.')
    df['state_time'] = pd.to_datetime(df['state_time'], utc=True)
    if df.duplicated(['underlying', 'state_time']).any():
        raise RuntimeError('Duplicate asset/timestamp rows.')
    print(f'rows {len(df):,} | assets {sorted(df.underlying.unique())} | {df.state_time.min()} -> {df.state_time.max()}')
    return df.reset_index(drop=True)

def schema_columns():
    con = duckdb.connect(str(DB_PATH), read_only=True)
    try:
        return con.execute('DESCRIBE time_series_feature_state').df()
    finally:
        con.close()

df = load_frame(['futures_index_price'], START, END, ASSETS)
df = df[df.state_time < ow.VRP_CONFIRMATION_START].reset_index(drop=True)
grid = vb.build_grid(df)
jres = jc.run(grid)
excluded = jres['ticks'].loc[jres['ticks'].tick_artifact, 'underlying'].tolist()
print('Module 09 P2 tick-artifact exclusions:', excluded)
panel, S, Z, J, valid = jres['panel'], jres['S'], jres['Z'], jres['J'], jres['valid']
BVb = ow.backward_bipower(panel, S)
sess_bar = ow.session_of_bar(panel)
used = [a for a in panel.assets if a not in excluded]
params = ow.loso_jump_params(panel, J, valid, Z, panel.R, sess_bar, used)
print('sessions with LOSO parameters:', len(params))


In [ ]:
con = duckdb.connect(str(DB_PATH), read_only=True)
cols = set(con.execute('DESCRIBE normalized_option_chain').df()['column_name'])
need = ['state_id', 'state_time', 'underlying', 'option_family_id', 'option_type', 'strike', 'best_bid_price',
        'best_ask_price', 'option_forward_price', 'minutes_to_expiry', 'instrument_type']
miss = sorted(set(need) - cols)
if miss:
    raise RuntimeError(f'normalized_option_chain lacks {miss}')
chain = con.execute(f"""
    WITH prim AS (SELECT DISTINCT state_id, option_family_id FROM primary_option_surface_feature_state)
    SELECT {', '.join('n.' + c for c in need)}
    FROM normalized_option_chain n
    JOIN prim p ON p.state_id = n.state_id AND p.option_family_id = n.option_family_id
    WHERE n.minutes_to_expiry BETWEEN {ow.MIN_TTE_MIN} AND {ow.MAX_TTE_MIN}
      AND EXTRACT(minute FROM n.state_time) % {ow.QUOTE_GRID_MIN} = 0
      AND n.state_time < TIMESTAMPTZ '{ow.VRP_CONFIRMATION_START.isoformat()}'
      AND ((lower(substr(n.option_type, 1, 1)) = 'c' AND n.strike > n.option_forward_price)
        OR (lower(substr(n.option_type, 1, 1)) = 'p' AND n.strike < n.option_forward_price))
""").df()
con.close()
chain = chain[chain.underlying.isin(used)]
quotes = ow.prepare_quotes(chain)
floors = ow.tick_floors(quotes)
display(floors)
t0 = time.time()
q = ow.attach_model(quotes, panel, S, BVb, params)
q = ow.flag_floor(q, floors)
print(f'priced {len(q):,} quotes in {time.time()-t0:.0f}s | floor-bound share {q.floor_bound.mean():.3f}')


In [ ]:
prim = ow.primary_sample(q, excluded)
primary = ow.session_median_stat(prim, 'log_mid_over_jump', n_boot=N_BOOT)
nojump = ow.session_median_stat(prim, 'log_mid_over_diff', n_boot=N_BOOT)
print('PRIMARY (z in [2,5)):', json.dumps(primary, indent=2, default=float))
print('same rows vs diffusion-only model:', json.dumps(nojump, indent=2, default=float))
sens = []
for label, ds, ls in [('diffusion_x0.85', 0.85, 1.0), ('lambda_x0.5', 1.0, 0.5), ('lambda_x2', 1.0, 2.0)]:
    x = prim.copy()
    x['lr'] = ow.reprice(prim, params, diff_scale=ds, lam_scale=ls)
    sens.append({'sensitivity': label, **ow.session_median_stat(x, 'lr', n_boot=N_BOOT)})
sens = pd.DataFrame(sens)
display(sens.round(4))
by_asset = pd.DataFrame([{'underlying': u, **ow.session_median_stat(g, 'log_mid_over_jump', n_boot=1000)}
                         for u, g in prim.groupby('underlying')])
display(by_asset.round(4))


In [ ]:
buckets = ow.bucket_table(q, n_boot=1000)
display(buckets.round(4))
piv = buckets.pivot(index='z_bucket', columns='tte_bucket', values='jump_M')
display(piv.round(3))
settle = ow.settlement_prices(grid, sorted(q.session.unique()))
pnl = ow.settlement_pnl(q, settle)
display(pnl.round(3))


In [ ]:
q.drop(columns=[c for c in ['instrument_name'] if c in q]).to_csv(OUT / '10_priced_quotes.csv.gz', index=False, compression='gzip')
floors.to_csv(OUT / '10_tick_floors.csv', index=False)
buckets.to_csv(OUT / '10_bucket_table.csv', index=False)
by_asset.to_csv(OUT / '10_primary_by_asset.csv', index=False)
sens.to_csv(OUT / '10_primary_sensitivities.csv', index=False)
pnl.to_csv(OUT / '10_settlement_pnl.csv', index=False)
pd.DataFrame([{'session': s, 'n_jumps_used': p['n_jumps_used'], **{f'lambda_{a}': v for a, v in p['lambda_bar'].items()}}
              for s, p in params.items()]).to_csv(OUT / '10_loso_jump_params.csv', index=False)
decision = {'primary_z_2_5': primary, 'same_rows_diffusion_only': nojump, 'excluded_assets_tick_artifact': excluded}
(OUT / '10_decision.json').write_text(json.dumps(decision, indent=2, default=float))
meta = {'version': ow.VERSION, 'module': '10_jump_model_vs_option_wings',
        'cross_branch_reuse': 'approved by Ivan 2026-10-10: VRP option_pricing conventions + VRP 006 audit findings',
        'vrp_confirmation_start_excluded': str(ow.VRP_CONFIRMATION_START), 'tte_min': [ow.MIN_TTE_MIN, ow.MAX_TTE_MIN],
        'quote_grid_min': ow.QUOTE_GRID_MIN, 'primary_z': list(ow.PRIMARY_Z), 'n_quotes_priced': int(len(q)),
        'n_primary_rows': int(len(prim)), 'first_time': str(df.state_time.min()), 'last_time': str(df.state_time.max())}
(OUT / '10_run_metadata.json').write_text(json.dumps(meta, indent=2, default=float))
print(json.dumps(decision, indent=2, default=float))
